# P5 — Granite on the LinuxONE itself

This notebook runs **on the team VM** (s390x). It:

1. checks what the VM offers (its Jupyter kernel already has `llama_cpp`, i.e.
   llama-cpp-python);
2. loads `granite-4.0-h-350m-base-be.f16.gguf` **inside this kernel**, with no server;
3. explains alerts with that Granite and shows what the validation accepted or rejected.

About the file: GGUF is llama.cpp's format; `-be` = big-endian (required on s390x); `f16` =
16-bit weights (~685 MB in memory); `h-350m` = Granite 4.0 hybrid with 350 million parameters;
`base` = **not** instruction-tuned. That is why the prompt is few-shot (worked examples), and
every text goes through the same validation: if the model invents anything, the deterministic
text comes back.

**The VM is shared (3 vCPUs and ~8 GB for the whole team):** 1 thread; run the last cell
(`close`) and shut the kernel down (*Running* tab) when you are done. The alerts here are
**mock**.

In [ ]:
import os
import sys
import time
from pathlib import Path

# Walk up from the notebook's folder to the clone (e.g. ~/shared/civicwater/users/<name>/repo).
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(REPO / "src"))

for v in ["OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"]:
    os.environ[v] = "1"

os.environ.setdefault("LOCAL_LLM_MODEL_PATH", str(Path.home() / "granite-4.0-h-350m-base-be.f16.gguf"))
os.environ.setdefault("LOCAL_LLM_THREADS", "1")

print("repo:", REPO)
print("kernel python:", sys.executable, sys.version.split()[0])
print("model:", os.environ["LOCAL_LLM_MODEL_PATH"])

## 1. What this VM has (read only)

In [ ]:
import glob
import importlib.util
import platform
import shutil
import struct

home = Path.home()
print("architecture:", platform.machine(), "| CPUs:", os.cpu_count())
try:
    meminfo = Path("/proc/meminfo").read_text().splitlines()
    print(*[l for l in meminfo if l.startswith(("MemTotal", "MemAvailable"))], sep="\n")
    cpuinfo = Path("/proc/cpuinfo").read_text()
    print("NNPA (Telum AI accelerator) visible:", " nnpa" in cpuinfo or "nnpa " in cpuinfo)
except OSError:
    pass

print("\nprograms:")
for b in ["llama-server", "llama-cli", "gcc", "cmake", "uv"]:
    print(f"  {b:13} {shutil.which(b)}")

print("\nPython packages in this kernel:")
for m in ["llama_cpp", "pydantic", "fastapi", "pytest"]:
    print(f"  {m:13} {'yes' if importlib.util.find_spec(m) else '-'}")

print("\n.gguf files:")
for p in sorted(set(glob.glob(str(home / "*.gguf")) + glob.glob(str(home / "*/*.gguf")))):
    with open(p, "rb") as f:
        head = f.read(8)
    if head[:4] != b"GGUF":
        print("  ", p, "-> not a GGUF file")
        continue
    le, be = struct.unpack("<I", head[4:]), struct.unpack(">I", head[4:])
    endian = "big-endian" if be[0] < le[0] else "little-endian"
    print(f"   {p}  ({os.path.getsize(p) / 1e6:.0f} MB, GGUF v{min(le[0], be[0])}, {endian})")

## 2. Load Granite in this kernel

It loads once (a few seconds) and stays in memory until the last cell (`close`). The
`system info` line shows what llama.cpp uses on the CPU (look for `VXE`, and `NNPA`/`ZDNN` if
the build has the accelerator). If this fails, copy the message: it may be a llama-cpp-python
version without support for the Granite 4.0 hybrid architecture (`granitehybrid`).

In [ ]:
import llama_cpp

from sentinel.narrative import LlamaCppNarrator, LocalLLMError

print("llama-cpp-python:", getattr(llama_cpp, "__version__", "?"))
try:
    info = llama_cpp.llama_print_system_info()
    print("system info:", info.decode() if isinstance(info, bytes) else info)
except Exception as exc:
    print("system info unavailable:", exc)

granite = LlamaCppNarrator.from_env()
t = time.perf_counter()
try:
    granite.load()
    print(f"model loaded in {time.perf_counter() - t:.1f} s:", granite.name)
except LocalLLMError as exc:
    print("did NOT load:", exc)

## 3. Explain an alert with the local Granite

`explain()` sends the model two invented examples **with the same shape as the alert** (with or
without a region) and the alert's facts. The text is accepted only if **every** sensor, region,
number, date and time in it comes from the alert and sits in the right place; otherwise the
deterministic text comes back, with the reason in `fallback_reason`. The `[SIMULATION]` notice
is added by the code, not by the model. `model wrote` shows Granite's raw text, even when it was
rejected.

In [ ]:
from sentinel.narrative import explain
from sentinel.narrative.local import build_completion_prompt
from sentinel.narrative.mock import mock_alert

alert = mock_alert()  # invented: region R4, CUSUM, 3 sensors
print("...", build_completion_prompt(alert)[-500:], "\n")

t = time.perf_counter()
e = explain(alert, granite)
print(f"time: {time.perf_counter() - t:.1f} s | source: {e.source} | fallback: {e.fallback_reason}")
print("tokens:", granite.last_usage)
print("model wrote:", e.model_text)
print("check:", e.check)
print("\nshown to the operator:\n" + e.text)

## 4. The same with a pipeline alert (the backend's mock detector)

In [ ]:
from datetime import UTC, datetime

from sentinel.audit import HashChainAuditSink
from sentinel.config import Settings
from sentinel.main import build_pipeline
from sentinel.schemas.windows import SensorWindow
from sentinel.sensors import SensorRegistry
from sentinel.synthetic import generate

registry = SensorRegistry.load(REPO / "config" / "sensors.provisional.json")
pipeline = build_pipeline(Settings(), registry=registry, audit_sink=HashChainAuditSink())
for w in generate(datetime(2019, 3, 14, tzinfo=UTC), 288, registry, seed=7, noise_m=0.05,
                  leak_at=200, leak_sensors=registry.pressure[:3]):
    pipeline.ingest(SensorWindow.model_validate(w))

pipeline_alert = pipeline.alerts.query(limit=1, offset=0)[0][0]
t = time.perf_counter()
e = explain(pipeline_alert, granite)
print(f"time: {time.perf_counter() - t:.1f} s | source: {e.source} | fallback: {e.fallback_reason}")
print("tokens:", granite.last_usage)
print("model wrote:", e.model_text)
print("check:", e.check)
print("shown to the operator:\n" + e.text)

## 5. How to read the result

- `source: llm`: Granite's text passed the validation and would be shown to the operator.
- `source: template` with `narrative_rejected`: the model invented or copied something (see
  `check`); the operator sees the deterministic text. With a 350M *base* model this can happen.
- `narrator_error`: the model did not load or failed (message in `fallback_reason`).

Write down the **load time** (section 2) and the **time per explanation** (sections 3 and 4):
explanations are generated in the background and stored (`sentinel.narrative.store`), so ~30 s
per alert is fine, but a fast replay builds a queue. If the acceptance rate is low, the
alternatives are a Granite *instruct* model converted to big-endian, or `ibm/granite-4-h-small`
on watsonx.ai (`WatsonxNarrator`).

From a terminal (after `source …/start.sh`): `python -m sentinel.narrative --llamacpp` (prints
`model_load_s`, `elapsed_s` and the token usage).

## 6. Free the memory

In [ ]:
granite.close()
print("model unloaded:", not granite.loaded)